# 2. Train, and compare against the reference approach

The reference repository's method is DTW alignment to a reference performance. It is implemented here properly and given every advantage: three distance functions, banded and unbanded warping, a canonical or a real exemplar reference, and a per-action isotonic calibration of its distance onto the quality scale. The configuration is selected on **validation**, never on test.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt

# Notebooks run with cwd=notebooks/, so the figure and table roots have to be
# repointed at the repository's results/ tree. Without this every figure lands in
# notebooks/results/figures/ and the committed figures silently never update.
import saqa.viz as _viz, saqa.report as _report
_viz.FIGURES = ROOT / "results" / "figures"
_viz.TABLES = _report.TABLES = ROOT / "results" / "tables"
tables = _viz.TABLES
from saqa.config import load_config
from saqa.pipelines import make_splits, run_single, fit_baselines
cfg = load_config('../configs/base.yaml', ['data.num_sequences=300', 'optim.epochs=4', 'run.out_dir=../results/runs_nb'])
splits = make_splits(cfg); splits.sizes()

{'train': 190, 'val': 34, 'test': 76}

## Per-action calibration is worth more than the alignment

A single global isotonic map conflates 'which action' with 'how good', because a throw and a gait cycle have different spatial extents. Fixing that is the single largest change to the baseline's score, and it is the difference between a strawman and a real comparison.

In [2]:
from saqa.baselines.fitted import DTWBaseline
from saqa.metrics import spearman
from saqa.pipelines import generator_config
g = generator_config(cfg)
rows = []
for per_action in (False, True):
    m = DTWBaseline(generator=g, per_action_calibration=per_action).fit(splits.train)
    s, _, _ = m.predict(splits.test)
    rows.append({'per_action_calibration': per_action,
                 'test_spearman': spearman(splits.test.quality, s)})
pd.DataFrame(rows).round(4)

,per_action_calibration,test_spearman
0,False,-0.0952
1,True,0.3804


## Train the graph model and the graph-free controls

In [3]:
runs = {}
for arch in ('saqa_stgcn', 'tcn', 'frame_average'):
    runs[arch] = run_single(cfg, name=f'nb_{arch}', architecture=arch,
                            splits=splits, save=False, verbose=False)
    print(f"{arch:16s} rho {runs[arch].metrics['spearman']:+.4f}  "
          f"params {runs[arch].metrics['params']:.0f}")

saqa_stgcn       rho +0.2107  params 66791


tcn              rho +0.2318  params 122707


frame_average    rho -0.2341  params 25965


In [4]:
base = fit_baselines(splits, cfg, dtw_sweep=False)
rows = [{'method': k, 'spearman': spearman(splits.test.quality, v.metrics['spearman'] if False else runs[k].pred['score'])} for k in runs]
for k in ('kinematic_gbr', 'dtw_reference'):
    rows.append({'method': k, 'spearman': spearman(splits.test.quality, base[k]['score'])})
pd.DataFrame(rows).sort_values('spearman', ascending=False).round(4)

,method,spearman
4,dtw_reference,0.3804
3,kinematic_gbr,0.3216
1,tcn,0.2318
0,saqa_stgcn,0.2107
2,frame_average,-0.2341


## The untrained control

A random-weights network is **not** a trivial baseline here: a random projection of movement amplitude already correlates with defect severity. Any trained model that does not clear this has learned nothing the architecture and the input statistics did not already provide.

In [5]:
from saqa.models import build_model
from saqa.engine import predict
torch.manual_seed(999)
u = predict(build_model('saqa_stgcn'), splits.test.coords)['score']
print(f'untrained saqa_stgcn: rho {spearman(splits.test.quality, u):+.4f}')

untrained saqa_stgcn: rho -0.2587


## Committed results (from `make all`)

In [6]:
for name in ('method_comparison', 'statistical_tests'):
    p = pathlib.Path('../results/tables') / f'{name}.csv'
    if p.exists():
        print(f'--- {name} ---'); display(pd.read_csv(p).round(4))

--- method_comparison ---


,method,family,spearman,kendall_tau,pearson,relative_l2,mae,rmse,tie_fraction_truth,n,...,aurc_oracle,e_aurc,error_auroc,best_epoch,best_val_spearman,train_seconds,n_train,n_val,n_test,params
0,saqa_stgcn,neural,0.3842,0.2641,0.3788,0.2498,0.1526,0.1878,0.14,250.0,...,0.0680,0.0742,0.5347,7.0,0.4408,265.5096,638.0,112.0,250.0,66791.0
1,stgcn_dense,neural,0.3507,0.2380,0.3406,0.2518,0.1541,0.1893,0.14,250.0,...,0.0704,0.0784,0.5086,7.0,0.4863,214.1942,638.0,112.0,250.0,194007.0
2,tcn,neural,0.3305,0.2281,0.3553,0.2521,0.1557,0.1896,0.14,250.0,...,0.0726,0.0712,0.5491,9.0,0.4491,41.5734,638.0,112.0,250.0,122707.0
3,lstm,neural,0.3569,0.2423,0.3683,0.2498,0.1561,0.1879,0.14,250.0,...,0.0742,0.0814,0.4985,9.0,0.4452,393.5391,638.0,112.0,250.0,159859.0
4,frame_average,neural,0.2381,0.1633,0.2665,0.2626,0.1600,0.1974,0.14,250.0,...,0.0717,0.0885,0.4971,6.0,0.2916,4.3153,638.0,112.0,250.0,25965.0
5,untrained_stgcn,control,-0.0547,-0.0358,-0.0223,0.6319,0.4353,0.4751,0.14,250.0,...,0.2629,0.1768,0.5552,NaN,NaN,NaN,NaN,NaN,NaN,66791.0
6,kinematic_gbr,baseline,0.5968,0.4271,0.5973,0.2140,0.1289,0.1609,0.14,250.0,...,0.0571,0.0681,0.5398,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,dtw_reference,baseline,0.4996,0.3467,0.4830,0.2407,0.1452,0.1810,0.14,250.0,...,0.0633,0.0852,0.4744,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,framewise_reference,baseline,0.4191,0.2950,0.4213,0.2524,0.1512,0.1898,0.14,250.0,...,0.0641,0.0870,0.5193,NaN,NaN,NaN,NaN,NaN,NaN,NaN


--- statistical_tests ---


,family,name_a,name_b,mean_a,mean_b,difference,ci_lower,ci_upper,p_value,p_adjusted,effect_size,n,significant
0,abs_error,saqa_stgcn.abs_error,dtw_reference.abs_error,0.1526,0.1452,0.0074,-0.0052,0.0202,0.2727,0.3908,0.0718,250,False
1,abs_error,stgcn_dense.abs_error,dtw_reference.abs_error,0.1541,0.1452,0.0089,-0.0037,0.0215,0.1288,0.3863,0.0879,250,False
2,abs_error,tcn.abs_error,dtw_reference.abs_error,0.1557,0.1452,0.0104,-0.0009,0.0220,0.0463,0.2779,0.1136,250,False
3,abs_error,lstm.abs_error,dtw_reference.abs_error,0.1561,0.1452,0.0109,0.0002,0.0223,0.0815,0.3262,0.1207,250,False
4,abs_error,frame_average.abs_error,dtw_reference.abs_error,0.1600,0.1452,0.0148,0.0007,0.0292,0.0624,0.3121,0.1335,250,False
5,abs_error,kinematic_gbr.abs_error,dtw_reference.abs_error,0.1289,0.1452,-0.0163,-0.0300,-0.0031,0.0150,0.1048,-0.1538,250,False
6,abs_error,framewise_reference.abs_error,dtw_reference.abs_error,0.1512,0.1452,0.0059,-0.0027,0.0148,0.1954,0.3908,0.0861,250,False
7,abs_error,untrained_stgcn.abs_error,dtw_reference.abs_error,0.4353,0.1452,0.2901,0.2635,0.3172,0.0000,0.0000,1.3089,250,True
8,spearman,saqa_stgcn.spearman,dtw_reference.spearman,0.3842,0.4996,-0.1154,-0.2301,-0.0092,NaN,NaN,NaN,2000,True
9,spearman,stgcn_dense.spearman,dtw_reference.spearman,0.3507,0.4996,-0.1489,-0.2689,-0.0373,NaN,NaN,NaN,2000,True
